# 测试集 + 迁移评估：base vs GRPO（lr 1e-4，seed 1 / seed 2）

最终报告用的评估。超参数已经在验证集上定好（lr 1e-4，1250 步），**这里的结果只用来报告，不能再拿来调参或挑 checkpoint**。

| 数据集 | 来源 | 题数 | 结果文件里的标记 |
|---|---|---|---|
| MedQA 测试集 | `GBaker/MedQA-USMLE-4-options` test split | 1273 | `medqa4` |
| MedMCQA（迁移） | `openlifescienceai/medmcqa` validation（测试集没有公开答案） | 约 4183 | `medmcqa` |
| PubMedQA（迁移） | `qiaojin/PubMedQA` pqa_labeled（专家标注） | 1000 | `pubmedqa` |

三种评估方式和验证集那份 notebook 完全相同（模型都是 **bf16 不量化**）：

| 方式 | 脚本 | confidence |
|---|---|---|
| `direct` | `eval/evaluate.py --quantization none` | 第一个 token 上各字母的概率 |
| `cot` | `eval/cot_eval.py`（GRPO 训练 prompt，贪心） | `Answer:` 后字母 token 的概率 |
| `cot_vote8` | 同上，采样 8 次（temperature 1.0） | 8 次里选这个答案的比例 |

PubMedQA 的 CoT prompt：在题目前加 `Context:` + 摘要，最后一句改成 `X is one of A, B, or C`，其余和训练 prompt 相同。MedQA / MedMCQA 的 prompt 和训练时一字不差。

按顺序运行：检查 GPU → 设置变量 → 拉取 repo → 安装依赖 →（如果 pip 提示）重启会话 → 从第 2 格（设置变量）往下运行，跳过第 4 格。重启后第 3 格跑不跑都行：第 5 格自己会 `import os, Path` 并切换到 repo 目录，后面的格子不用第 3 格定义的 `sh()`。运行时选 **A100**。

用时（按验证集 500 题约 5 分钟/模型粗估，没实测）：CoT 每个模型约 40–60 分钟，3 个模型共约 2–3 小时；direct 共约 10 分钟。可以在第 2 格 `DATASETS` 里只留 `'medqa'` 先跑 MedQA。
结果写到 Drive 的 `RESULTS_DIR`；已经跑完的评估会自动跳过，断线后重新从第 1 格开始运行即可。

In [ ]:
# 1. 检查 GPU 和 Python 版本
!nvidia-smi
import sys, torch
print('Python', sys.version.split()[0])
print('torch', torch.__version__, '| CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print('GPU:', torch.cuda.get_device_name(0), f'| compute {major}.{minor} →', 'bf16' if major >= 8 else 'fp16')
assert torch.cuda.is_available(), '没有 GPU：运行时 → 更改运行时类型 → GPU'
assert sys.version_info >= (3, 11), 'TRL 1.16 需要 Python >= 3.11'

In [ ]:
# 2. 设置变量
REPO_URL = 'https://github.com/skjdsk/medqa-calibration.git'
BRANCH   = 'main'
REPO_DIR = '/content/medqa-calibration'
TRL_VERSION = '1.16.0'
USE_VLLM = True  # 第 4 格按训练时同样的版本装 vLLM

DRIVE_ROOT = '/content/drive/MyDrive/medqa-calibration'
# 要评估的 adapter：{模型名: adapter 目录}；base 不需要 adapter
ADAPTERS = {
    'grpo_lr1e-4_seed1': f'{DRIVE_ROOT}/grpo/grpo_train5000_g8_ep1_len1024_lr0.0001_seed1_bf16/adapter',
    'grpo_lr1e-4_seed2': f'{DRIVE_ROOT}/grpo/grpo_train5000_g8_ep1_len1024_lr0.0001_seed2_bf16/adapter',
}
RESULTS_DIR = f'{DRIVE_ROOT}/eval_test_grpo'   # 和验证集结果（eval_val）分开放
DATASETS    = ['medqa', 'medmcqa', 'pubmedqa'] # 想先只跑 MedQA 测试集就改成 ['medqa']
N_SAMPLES   = 8      # cot_vote 的采样次数（和训练时每题 8 个回答一致）
N_DEBUG     = None   # 先想试几题就改成 20（会写进同一个目录，正式跑之前要删掉那些 CSV）；正式用 None

In [ ]:
# 3. git clone / pull repo
# 私有 repo：在 Colab 左侧 🔑（Secrets）里添加 GITHUB_TOKEN（GitHub 个人访问令牌），并打开"笔记本访问"
import os, subprocess
from pathlib import Path

def sh(cmd, shown=None):
    print('$', shown or cmd)
    subprocess.run(cmd, shell=True, check=True)

clone_url = REPO_URL
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
    if gh_token and REPO_URL.startswith('https://github.com/'):
        clone_url = REPO_URL.replace('https://', f'https://{gh_token}@')
except Exception:
    pass  # 没设置 GITHUB_TOKEN 就按公开 repo 处理

if (Path(REPO_DIR) / '.git').exists():
    sh(f'git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {clone_url} {REPO_DIR}', shown=f'git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}')
os.chdir(REPO_DIR)
sh('git log -1 --oneline')
for f in ['eval/evaluate.py', 'eval/cot_eval.py', 'eval/metrics.py', 'train/grpo/grpo_data.py', 'train/grpo/rewards.py', 'data/medqa_split_seed42.json']:
    assert Path(f).exists(), f'repo 里找不到 {f}：确认文件已经 push 到 {BRANCH}'
print('cwd =', os.getcwd())

In [ ]:
# 4. 安装依赖
from pathlib import Path
ip = get_ipython()
if Path('requirements.txt').exists():
    ip.run_line_magic('pip', 'install -q -r requirements.txt')
ip.run_line_magic('pip', f'install -q "trl[peft,quantization]=={TRL_VERSION}" "matplotlib" "pandas" "pytest"')
if USE_VLLM:
    # vLLM 会安装它自己需要的 torch 版本，装完后通常要 运行时 → 重启会话，再从"设置变量"那格重新运行
    # vLLM 0.28+ 删掉了 bitsandbytes 量化，而 TRL 在 QLoRA（4-bit）时会让 vLLM 用 bitsandbytes 加载模型
    # → 固定 0.27.1（最后一个支持 bitsandbytes 的版本，且在 TRL 1.16 允许的 0.22–0.31 范围内，要求 torch==2.13.0）
    ip.run_line_magic('pip', f'install -q "trl[vllm]=={TRL_VERSION}" "vllm==0.27.1"')

# Colab 预装的 torchao 0.10 太旧：peft 0.21 在不量化（--quantization none）加 LoRA 时会直接报错
# "only versions above 0.16.0 are supported"。我们用不到 torchao（vllm / peft 都不依赖它），直接卸载
ip.system('pip uninstall -y -q torchao')

import importlib.metadata as md
for pkg in ['torch', 'transformers', 'trl', 'peft', 'bitsandbytes', 'accelerate', 'datasets'] + (['vllm'] if USE_VLLM else []):
    try:
        print(f'{pkg:14s}', md.version(pkg))
    except md.PackageNotFoundError:
        print(f'{pkg:14s} 未安装')
print('\n如果上面 pip 提示需要重启（或之后 import 报错）：运行时 → 重启会话，然后从"设置变量"那格重新运行。')

**装完依赖后：菜单 运行时 → 重启会话**，然后从第 2 格（设置变量）开始往下运行（跳过第 4 格）。

In [ ]:
# 5. 挂载 Google Drive，检查 adapter 是否存在
import os
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
os.chdir(REPO_DIR)
for name, path in ADAPTERS.items():
    ok = Path(path, 'adapter_config.json').exists()
    print(f'{name:12s}', '✅' if ok else '❌ 找不到', path)
    assert ok, f'{name} 的 adapter 不存在：检查路径，或等 Drive 同步完'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('结果目录：', RESULTS_DIR)

In [ ]:
# 6.（可选）设置 HF_TOKEN：Qwen2.5 和 MedQA 都是公开的，不设也能下载；设置后下载更稳定、不容易被限速
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('已设置 HF_TOKEN')
except Exception:
    print('没有找到 HF_TOKEN（可选），继续匿名下载')

In [ ]:
# 7. 单元测试（几秒钟）
!python -m pytest -q eval/test_cot_eval.py eval/test_metrics.py train/grpo/test_rewards.py

In [ ]:
# 8. 运行评估的小工具：结果文件已经存在就跳过（断线重跑不会重复花时间）
import subprocess
N_ARG = f' --n {N_DEBUG}' if N_DEBUG else ''

def run_eval(cmd, expected_csv):
    if Path(RESULTS_DIR, expected_csv).exists():
        print('已存在，跳过：', expected_csv)
        return
    print('$', cmd)
    get_ipython().system(f'{cmd} 2>&1 | tee -a {RESULTS_DIR}/eval_log.txt')
    assert Path(RESULTS_DIR, expected_csv).exists(), f'没有生成 {expected_csv}：看上面的报错'

MODELS = {'base': None, **ADAPTERS}
# 数据集 → (命令行参数, 结果文件名里的标记)；标记和 evaluate.py / cot_eval.py 的命名一致
DS_ARGS = {
    'medqa':    ('--dataset medqa --medqa_subset test', 'medqa4'),
    'medmcqa':  ('--dataset medmcqa', 'medmcqa'),
    'pubmedqa': ('--dataset pubmedqa', 'pubmedqa'),
}
TAGS = [DS_ARGS[d][1] for d in DATASETS]
print('模型：', list(MODELS)); print('数据集：', TAGS)

In [ ]:
# 9. 直接作答（letter-logit），每个模型 × 数据集一个进程；MedQA 不加 --medqa_subset = 官方 test split
for name, adapter in MODELS.items():
    ad = f' --adapter {adapter}' if adapter else ''
    for ds in DATASETS:
        run_eval(f'python -m eval.evaluate --dataset {ds} --quantization none'
                 f'{ad} --model_name {name} --out_dir {RESULTS_DIR}{N_ARG}',
                 f'preds_{name}_{DS_ARGS[ds][1]}_direct.csv')

In [ ]:
# 10. 推理后作答（GRPO 训练 prompt，vLLM）。每个模型 × 每个数据集单独起一个进程（vLLM 退出后显存才完全释放）
for ds in DATASETS:
    ds_arg, tag = DS_ARGS[ds]
    for name, adapter in MODELS.items():
        ad = f' --adapter {adapter}' if adapter else ''
        run_eval(f'python -m eval.cot_eval {ds_arg}{ad} --model_name {name} --n_samples {N_SAMPLES} '
                 f'--out_dir {RESULTS_DIR}{N_ARG}',
                 f'preds_{name}_{tag}_cot_vote{N_SAMPLES}.csv' if N_SAMPLES else f'preds_{name}_{tag}_cot.csv')

In [ ]:
# 11. 汇总：准确率、ECE、Brier、NLL 等（eval/metrics.py）
import pandas as pd
from IPython.display import display, Image

MODES = ['direct', 'cot', f'cot_vote{N_SAMPLES}']
# 只汇总已经存在的文件（某些还没跑完也不会报错）
found = [str(f) for t in TAGS for m in MODES for f in sorted(Path(RESULTS_DIR).glob(f'preds_*_{t}_{m}.csv'))]
print(f'找到 {len(found)} 个结果文件')
files = ' '.join(f'"{f}"' for f in found)
!python -m eval.metrics --pred_csv {files} --out_dir {RESULTS_DIR}/metrics > /dev/null
s = pd.read_csv(f'{RESULTS_DIR}/metrics/summary.csv')
cols = ['dataset', 'eval_mode', 'model_name', 'n', 'accuracy', 'mean_confidence', 'ece', 'brier', 'nll',
        'hc_wrong_rate', 'mean_conf_wrong', 'mean_letter_mass']
display(s[cols].sort_values(['dataset', 'eval_mode', 'model_name']).round(4))
# 注意：cot_vote 的 NLL 不可靠 —— 8 票都没投正确答案时 p=0，被截到 1e-12，这道题的 NLL 就是 27.6

# cot 的额外信息：格式、截断、长度；cot_vote 的采样正确率
for mode, extra in [('cot', ['format_ok', 'truncated', 'n_completion_tokens']), (f'cot_vote{N_SAMPLES}', ['sample_acc', 'format_ok'])]:
    paths = [Path(RESULTS_DIR, f'preds_{m}_{t}_{mode}.csv') for t in TAGS for m in MODELS]
    parts = [pd.read_csv(q) for q in paths if q.exists()]
    if parts:
        print(mode); display(pd.concat(parts).groupby(['dataset', 'model_name'])[extra].mean().round(4))

In [ ]:
# 12. 和 base 的配对比较（同一道题对齐），每个 GRPO 模型 × 数据集 × 评估方式
#   - 改对 / 改错题数，粗略显著线 ±2×sqrt(两者之和)（符号检验的近似）
#   - 配对 bootstrap（对题目有放回重抽 2000 次）：准确率、平均 confidence、ECE 的差值（GRPO − base）的 95% 区间
#     区间不包含 0 → 这个差异在这批题上比较可靠
import numpy as np
from eval.metrics import prob_matrix, expected_calibration_error

def conf_correct(df):
    probs, gold = prob_matrix(df)
    return probs.max(axis=1), (probs.argmax(axis=1) == gold).astype(float)

rng = np.random.default_rng(0)
rows = []
for t in TAGS:
    for mode in MODES:
        p = Path(RESULTS_DIR, f'preds_base_{t}_{mode}.csv')
        if not p.exists():
            continue
        b = pd.read_csv(p).set_index('question_id')
        cb, kb = conf_correct(b)
        for name in ADAPTERS:
            q = Path(RESULTS_DIR, f'preds_{name}_{t}_{mode}.csv')
            if not q.exists():
                continue
            g = pd.read_csv(q).set_index('question_id')
            assert set(g.index) == set(b.index), f'{name} {t} {mode} 和 base 的题目不一致'
            g = g.loc[b.index]
            cg, kg = conf_correct(g)
            lost, gained = int(((kb == 1) & (kg == 0)).sum()), int(((kb == 0) & (kg == 1)).sum())
            n = len(b)
            boot = {'acc': [], 'conf': [], 'ece': []}
            for _ in range(2000):
                i = rng.integers(0, n, n)
                boot['acc'].append(kg[i].mean() - kb[i].mean())
                boot['conf'].append(cg[i].mean() - cb[i].mean())
                boot['ece'].append(expected_calibration_error(cg[i], kg[i], 15) - expected_calibration_error(cb[i], kb[i], 15))
            r = {'dataset': t, 'mode': mode, 'model': name, 'n': n, 'same_pred': round((b.pred == g.pred).mean(), 3),
                 'lost': lost, 'gained': gained, 'net': gained - lost, 'sig_line': round(2 * (lost + gained) ** 0.5)}
            for k, point in [('acc', kg.mean() - kb.mean()), ('conf', cg.mean() - cb.mean()),
                             ('ece', expected_calibration_error(cg, kg, 15) - expected_calibration_error(cb, kb, 15))]:
                lo, hi = np.percentile(boot[k], [2.5, 97.5])
                r[f'd_{k}'] = f'{point:+.3f} [{lo:+.3f}, {hi:+.3f}]'
            rows.append(r)
paired_df = pd.DataFrame(rows)
paired_df.to_csv(f'{RESULTS_DIR}/metrics/paired_vs_base.csv', index=False)
with pd.option_context('display.max_colwidth', None, 'display.width', 250):
    display(paired_df)

In [ ]:
# 13. 可靠性图（柱子低于对角线 = 过度自信）
for f in sorted(Path(RESULTS_DIR, 'metrics').glob('reliability_*.png')):
    print(f.name); display(Image(str(f), width=420))

In [ ]:
# 14. 强制把 Drive 上的结果写完再关机（不跑这格，运行时被回收时最后几个文件可能没上传）
from google.colab import drive
drive.flush_and_unmount()
print('全部上传完成')

**怎么读结果**

- 主要结论看 **MedQA 测试集 `direct`**（proposal 的主评估），`cot` / `cot_vote8` 是次要评估。
- 准确率：`net` 超过 `sig_line`，或 `d_acc` 的区间不包含 0，才算有变化。
- 校准：`d_conf` 为正且 `d_ece` 为正（区间不含 0）→ GRPO 让模型更自信、校准更差；再看 `hc_wrong_rate`（confidence ≥ 0.9 且答错）。
- 两个 seed 方向一致才写成结论；只有一个 seed 显著的，如实写"不稳定"。
- MedMCQA / PubMedQA 是迁移评估：模型只在 MedQA 上训练过。PubMedQA 的 CoT prompt 稍有改动（见最上面），`format_ok` 低的话先看生成内容（`generations_*_pubmedqa_cot.jsonl`）。